In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnan
import pandas as pd

spark = SparkSession.builder \
    .appName("Comprehensive_EDA_Project") \
    .getOrCreate()

hdfs_path = "hdfs://localhost:9000/user/bigdata/data/e-commerce_shopper_behaviour_and_lifestyle.csv"
df = spark.read.csv(hdfs_path, header=True, inferSchema=True)

In [2]:
# --- Kiểm tra cấu trúc và quy mô ---
print("Cấu trúc dữ liệu (Schema):")
df.printSchema()

print(f"Dataset có tổng cộng: {df.count()} dòng và {len(df.columns)} cột.")

Cấu trúc dữ liệu (Schema):
root
 |-- user_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- urban_rural: string (nullable = true)
 |-- income_level: integer (nullable = true)
 |-- employment_status: string (nullable = true)
 |-- education_level: string (nullable = true)
 |-- relationship_status: string (nullable = true)
 |-- has_children: integer (nullable = true)
 |-- household_size: integer (nullable = true)
 |-- occupation: string (nullable = true)
 |-- ethnicity: string (nullable = true)
 |-- language_preference: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- weekly_purchases: integer (nullable = true)
 |-- monthly_spend: integer (nullable = true)
 |-- cart_abandonment_rate: integer (nullable = true)
 |-- review_writing_frequency: integer (nullable = true)
 |-- average_order_value: integer (nullable = true)
 |-- preferred_payment_method: string (nullable = tru

In [2]:
#Check missing values
from pyspark.sql.functions import col, count, when, isnan

null_checks = [
    count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
    if df.schema[c].dataType.simpleString() in ['float', 'double']
    else count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]

print("Đang tìm giá trị bị thiếu...")
null_counts_raw = df.select(null_checks).toPandas()

null_report = null_counts_raw.transpose().reset_index()
null_report.columns = ['Tên cột', 'Số lượng Null']
null_report['Tỷ lệ (%)'] = (null_report['Số lượng Null'] / df.count() * 100).round(4)

display(null_report)

Đang tìm giá trị bị thiếu...


,Tên cột,Số lượng Null,Tỷ lệ (%)
0,user_id,0,0.0
1,age,0,0.0
2,gender,0,0.0
3,country,0,0.0
4,urban_rural,0,0.0
5,income_level,0,0.0
6,employment_status,0,0.0
7,education_level,0,0.0
8,relationship_status,0,0.0
9,has_children,0,0.0


In [3]:
# --- Kiểm tra dữ liệu trùng lặp ---
duplicate_count = df.count() - df.dropDuplicates(["user_id"]).count()
print(f"Số lượng bản ghi trùng lặp (dựa trên user_id): {duplicate_count}")

Số lượng bản ghi trùng lặp (dựa trên user_id): 0


In [5]:
#Check outliers các cột quan trọng
outlier_cols = [
    # Nhân khẩu học
    "age", "income_level", "household_size",
    # Hành vi mua sắm
    "monthly_spend", "average_order_value", "weekly_purchases",
    "cart_abandonment_rate", "purchase_conversion_rate",
    "impulse_purchases_per_month", "return_frequency",
    "checkout_abandonments_per_month",
    # Hành vi app/web
    "daily_session_time_minutes", "product_views_per_day",
    "ad_clicks_per_day", "wishlist_items_count",
    # Score
    "brand_loyalty_score", "impulse_buying_score",
    "social_media_influence_score", "mental_health_score",
    "overall_stress_level"
]

df.select(outlier_cols).summary("min", "25%", "75%", "max").show()

+-------+---+------------+--------------+-------------+-------------------+----------------+---------------------+------------------------+---------------------------+----------------+-------------------------------+--------------------------+---------------------+-----------------+--------------------+-------------------+--------------------+----------------------------+-------------------+--------------------+
|summary|age|income_level|household_size|monthly_spend|average_order_value|weekly_purchases|cart_abandonment_rate|purchase_conversion_rate|impulse_purchases_per_month|return_frequency|checkout_abandonments_per_month|daily_session_time_minutes|product_views_per_day|ad_clicks_per_day|wishlist_items_count|brand_loyalty_score|impulse_buying_score|social_media_influence_score|mental_health_score|overall_stress_level|
+-------+---+------------+--------------+-------------+-------------------+----------------+---------------------+------------------------+---------------------------+-